# I Gave My AI Less to Read, and It Did Better

A customer asks why their shipment is late. We have an SOP document, 80 hub scans and 3 long customer emails. Scan 11 holds the only line that matters: a **hold at Nagpur for an e-way bill mismatch**.

The AI can only read so much at once (here, 400 words). We pack what it reads three ways and ask two questions: *why is it late?* and *what did we promise?*

Run the cell below.

In [ ]:
# Lesson 04 — Context engineering: does more information make the model better or worse?
# Toy shipment assistant, standard library only. Run: python notebook.py
import random
random.seed(42)
hubs = ["Pune", "Nashik", "Nagpur", "Raipur", "Kolkata"]
sop = "Standard operating procedure: " + "check seals, scan labels, log weights, " * 40
events = [f"{random.choice(hubs)} hub: scanned, bagged and moved on schedule." for _ in range(80)]
events[10] = "Nagpur hub: HOLD. Consignment stopped due to e-way bill mismatch."
emails = ["Customer email 1: When will it arrive? Your team promised delivery by 26 Sep. " + "Please update us soon. " * 12,
          "Customer email 2: Still waiting, no update. " + "This is urgent for our plant. " * 12,
          "Customer email 3: If it is late, can we get a refund option? " + "Kindly confirm today. " * 12]
summary = "Summary: customer emailed 3 times; team promised delivery by 26 Sep; customer now asks about a refund."

WINDOW = 400  # the model's "desk": how many words fit. Older words fall off the front.
def model(context, question):  # toy MODEL: can only use facts it can still see
    visible = " ".join(context.split()[-WINDOW:])
    if question == "why late":
        return "e-way bill hold at Nagpur" if "HOLD" in visible else "No issue found, on track"
    return "delivery by 26 Sep" if "promised delivery by 26 Sep" in visible else "No record of a promise"

truth = {"why late": "e-way bill hold at Nagpur", "what did we promise": "delivery by 26 Sep"}
exceptions = [e for e in events if "HOLD" in e or "DELAY" in e]
packs = {  # three ways to pack the same model's desk
    "A: everything": " ".join([sop] + events + emails),
    "B: relevant only": " ".join(exceptions + events[-3:]),
    "C: relevant + summary": " ".join(exceptions + events[-3:] + [summary]),
}
PRICE = 3.0  # example price: $ per million input tokens
for name, ctx in packs.items():
    words = len(ctx.split())
    tokens = int(words * 1.3)  # rough rule: 1 English word is about 1.3 tokens
    monthly = tokens * 2 * 10_000 * 30 * PRICE / 1e6  # 2 questions, 10,000 shipments a day
    print(f"\n{name}  |  {words} words, ~{tokens} tokens, ~${monthly:,.0f}/month")
    for q, right in truth.items():
        ans = model(ctx, q)
        verdict = "RIGHT" if ans == right else ("SAYS IT DOESN'T KNOW" if "No record" in ans else "WRONG (and confident)")
        print(f"   {q:22} -> {ans:28} {verdict}")


## 5-minute tweak challenge

1. Set `WINDOW = 5000` so everything fits. Does pack A now get both right? What does it still cost?
2. Move the HOLD event to `events[75]` (near the end). Why does pack A suddenly "work"?
3. Delete the summary from pack C. Which answer breaks, and is that failure safe or dangerous?